In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.svm import SVR, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.linear_model import LassoCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [3]:
#load the features
features_df = pd.read_csv("/Users/folasewaabdulsalam/Downloads/EEG_Based_Mental_Workload_Classifier/EEG-Based_Mental_Workload_Classifier/features.csv")
print(f"The Features shape:{features_df.shape}")
features_df.head(5)

The Features shape:(93, 58)


,delta_AF3,delta_F7,delta_F3,delta_FC5,delta_T7,delta_P7,delta_O1,delta_O2,delta_P8,delta_T8,...,beta_O1,beta_O2,beta_P8,beta_T8,beta_FC6,beta_F4,beta_F8,beta_AF4,subject,workload
0,10.725343,5.077131,2.238050,0.697380,56.521224,15.568467,3.971748,1.247739,5.578479,1.934691,...,30.620624,9.059456,2.534956,1.105544,12.393469,5.364822,1.685556,0.879012,sub01,hi
1,5516.791766,963.839271,128.723981,12.964473,1396.342661,220.522152,44.662870,6.670238,5436.258042,945.965484,...,5296.014472,923.484204,129.758557,13.410973,5471.254522,949.507594,131.673970,14.620083,sub01,lo
2,3.989949,2.608310,1.053837,0.753365,19.472834,10.321517,4.888012,3.832464,4.447023,2.039295,...,18.877661,9.902051,3.861640,2.306138,3.648893,2.466794,0.987598,0.551459,sub02,hi
3,2.253481,0.975757,0.694703,0.259047,2.421729,0.950285,0.583619,0.401876,8.895102,1.115093,...,4.412904,1.214817,0.641196,0.527638,2.726505,0.995539,0.698546,0.281283,sub02,lo
4,55.389697,9.962111,4.743549,0.646834,103.442932,12.132124,5.344133,0.949898,8.043204,3.059159,...,87.510300,12.516137,6.051994,1.143680,26.308322,3.320430,3.043306,0.462703,sub03,hi


In [4]:
#load and merge ratings
ratings = pd.read_csv("/Users/folasewaabdulsalam/Downloads/EEG_Based_Mental_Workload_Classifier/EEG-Based_Mental_Workload_Classifier/ratings.txt", header=None, names=['subject', 'rating_low', 'rating_high'])
#create subject mapping to align with our features dataframe 1 → 'sub01'

subject_mapping = {i: f'sub{i:02d}' for i in range(1,49)}
ratings['subject'] = ratings['subject'].replace(subject_mapping)
print(ratings.head())

  subject  rating_low  rating_high
0   sub01           2            8
1   sub02           1            5
2   sub03           1            5
3   sub04           2            5
4   sub06           4            7


In [5]:
#merging the two datasets

#prepare the ratings in long format
ratings_long = []

for _, row in ratings.iterrows():
    ratings_long.append(
        {'subject': row['subject'], 'workload': 'lo', 'rating': row['rating_low']} #low workload ratings

    )
    ratings_long.append(
        {'subject': row['subject'], 'workload': 'hi', 'rating':row['rating_high']} #high workload ratings
    )
ratings_df = pd.DataFrame(ratings_long)
ratings_df.head()

#merging the long format of the ratings text with the features


data = features_df.merge(ratings_df, on=['subject', 'workload'], how='inner')

print(f"\nMerged data shape: {data.shape}")
print(data['rating'].value_counts().sort_index())
data.head()



Merged data shape: (87, 59)
rating
1    23
2    11
3     7
4     5
5    11
6     6
7    11
8    10
9     3
Name: count, dtype: int64


,delta_AF3,delta_F7,delta_F3,delta_FC5,delta_T7,delta_P7,delta_O1,delta_O2,delta_P8,delta_T8,...,beta_O2,beta_P8,beta_T8,beta_FC6,beta_F4,beta_F8,beta_AF4,subject,workload,rating
0,10.725343,5.077131,2.238050,0.697380,56.521224,15.568467,3.971748,1.247739,5.578479,1.934691,...,9.059456,2.534956,1.105544,12.393469,5.364822,1.685556,0.879012,sub01,hi,8
1,5516.791766,963.839271,128.723981,12.964473,1396.342661,220.522152,44.662870,6.670238,5436.258042,945.965484,...,923.484204,129.758557,13.410973,5471.254522,949.507594,131.673970,14.620083,sub01,lo,2
2,3.989949,2.608310,1.053837,0.753365,19.472834,10.321517,4.888012,3.832464,4.447023,2.039295,...,9.902051,3.861640,2.306138,3.648893,2.466794,0.987598,0.551459,sub02,hi,5
3,2.253481,0.975757,0.694703,0.259047,2.421729,0.950285,0.583619,0.401876,8.895102,1.115093,...,1.214817,0.641196,0.527638,2.726505,0.995539,0.698546,0.281283,sub02,lo,1
4,55.389697,9.962111,4.743549,0.646834,103.442932,12.132124,5.344133,0.949898,8.043204,3.059159,...,12.516137,6.051994,1.143680,26.308322,3.320430,3.043306,0.462703,sub03,hi,5


In [ ]:
#Data splitting an dapplying NCA feature selection

unique_subjects = np.unique(data['subject'])
n_subjects = len(unique_subjects)

train_subjects, test_subjects = train_test_split(n_subjects, test_size=0.2, random_state=42)#splitting by subjects

#creating masks to select th erecordings based on the subjecet split
train_mask = np.isin(data['subject'], train_subjects)
test_mask = np.isin(data['subject'], test_subjects)

X = data.drop(['subject', 'workload', 'rating'], axis=1)  
y = data['rating']  

#splitting features and targets


45
